
## Simple Gen AI APP Using Langchain 


In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 

##Langsmith Tracing
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [17]:
## Data Ingestion --From the website we need to scrape the data 
from langchain_community.document_loaders import WebBaseLoader

loader=WebBaseLoader("https://docs.langchain.com/langsmith/home")
loader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [18]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.\nIt helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent sta

In [19]:
## Load Data ---> Docs---> Divide our text into chunks --> text -->vectors--> vector Embeddings ---> Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [20]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.'),
 Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'languag

In [21]:
## Converting this all text into Vectors
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [22]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)


In [23]:
vectorstoredb

In [24]:
## Query from the Vector Store DB
query="LangSmith is a framework-agnostic platform for developing, debugging"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.'

In [25]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

In [26]:
## Retrival Chain, Document chain 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate 

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on provided context.
<context>
{context}
</context>

"""
)
documents_chain=create_stuff_documents_chain(llm,prompt)
documents_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on provided context.\n<context>\n{context}\n</context>\n\n'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=

In [27]:
from langchain_core.documents import Document
documents_chain.invoke({
    "input": "LangSmith is a framework-agnostic platform for developing, debugging",
    "context": [Document(page_content="LangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications. It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whether you're using an existing framework or building your own. Prototype locally, then move to production with integrated monitoring and evaluation to build more reliable AI agents.")]
})

'LangSmith is a versatile platform designed to aid in the development, debugging, and deployment of AI agents and LLM applications. It offers tools for tracing requests, evaluating outputs, testing prompts, and managing deployments, all within a single platform. LangSmith supports any agent stack, making it compatible with both existing frameworks and custom-built solutions. It allows for local prototyping before transitioning to production, with features for integrated monitoring and evaluation to enhance the reliability of AI agents.'

In [28]:
### Input---> Retriver--->vectorstoredb
vectorstoredb

In [29]:
vectorstoredb.as_retriever()

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x16a168260>, search_kwargs={})

In [ ]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,documents_chain)

ImportError: cannot import name 'create_retriever_chain' from 'langchain_classic.chains' (/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/langchain_classic/chains/__init__.py)